In [1]:
import os
os.environ['HF_HOME']=r'F:\Langchain\models'

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
import requests

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7048\2272844035.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [5]:
llm=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=100
)

model =ChatHuggingFace(llm=llm)

In [6]:
llm=model

In [7]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=03ff27140ac05416816f8d61f561fa56&query={city}'

  response = requests.get(url)

  return response.json()

In [8]:
from langchain.agents import create_agent

In [9]:
agent = create_agent(
    model=llm,
    tools=[get_weather_data],
    system_prompt="""
You are a helpful assistant.

When a user asks for information that requires multiple steps,
perform the steps in the correct order.

If the user asks for the weather of the capital of a state:
1. First determine the capital.
2. Then call the weather tool using that capital city.
3. Do not guess or substitute another city.
4. After receiving the tool result, provide the final answer.
"""
)

In [10]:
response = agent.invoke({
    "messages": [
        (
            "user",
            "Find the capital of Madhya Pradesh, then find its current weather condition."
        )
    ]
})

In [11]:
print(response["messages"][-1].content)

The capital of Madhya Pradesh is Bhopal. The current weather condition in Bhopal is light rain shower with a temperature of 25°C.


In [12]:
for i, message in enumerate(response["messages"]):
    print(f"\n========== MESSAGE {i} ==========")
    print("TYPE:", type(message).__name__)
    print("CONTENT:", message.content)


========== MESSAGE 0 ==========
TYPE: HumanMessage
CONTENT: Find the capital of Madhya Pradesh, then find its current weather condition.

========== MESSAGE 1 ==========
TYPE: AIMessage
CONTENT: 

========== MESSAGE 2 ==========
TYPE: ToolMessage
CONTENT: Error invoking tool 'get_weather_data' with kwargs {} with error:
 city: Field required
 Please fix the error and try again.

========== MESSAGE 3 ==========
TYPE: AIMessage
CONTENT: 

========== MESSAGE 4 ==========
TYPE: ToolMessage
CONTENT: {"request": {"type": "City", "query": "Bhopal, India", "language": "en", "unit": "m"}, "location": {"name": "Bhopal", "country": "India", "region": "Madhya Pradesh", "lat": "23.267", "lon": "77.400", "timezone_id": "Asia/Kolkata", "localtime": "2026-08-23 15:56", "localtime_epoch": 1787500560, "utc_offset": "5.50"}, "current": {"observation_time": "10:26 AM", "temperature": 25, "weather_code": 353, "weather_icons": ["https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0009_light

In [13]:
for i, message in enumerate(response["messages"]):
    if hasattr(message, "tool_calls") and message.tool_calls:
        print(f"\n========== TOOL CALL {i} ==========")

        for tool_call in message.tool_calls:
            print("Tool:", tool_call["name"])
            print("Arguments:", tool_call["args"])
            print("Tool Call ID:", tool_call["id"])


========== TOOL CALL 1 ==========
Tool: get_weather_data
Arguments: {}
Tool Call ID: call_746gm2eRfsRJ7F22rh0iHGKz

========== TOOL CALL 3 ==========
Tool: get_weather_data
Arguments: {'city': 'Bhopal'}
Tool Call ID: call_LDv2j7UB7XBQzeV5YerWKSW1


In [14]:
for i, message in enumerate(response["messages"]):

    print(f"\n{'=' * 60}")
    print(f"STEP {i}")
    print(f"{'=' * 60}")

    if type(message).__name__ == "HumanMessage":
        print("USER:")
        print(message.content)

    elif type(message).__name__ == "AIMessage":

        if message.tool_calls:
            print("LLM → TOOL")

            for tool_call in message.tool_calls:
                print(f"Tool: {tool_call['name']}")
                print(f"Arguments: {tool_call['args']}")

        else:
            print("LLM → FINAL ANSWER")
            print(message.content)

    elif type(message).__name__ == "ToolMessage":
        print("TOOL → LLM")
        print(f"Tool: {message.name}")
        print(f"Result: {message.content}")


STEP 0
USER:
Find the capital of Madhya Pradesh, then find its current weather condition.

STEP 1
LLM → TOOL
Tool: get_weather_data
Arguments: {}

STEP 2
TOOL → LLM
Tool: get_weather_data
Result: Error invoking tool 'get_weather_data' with kwargs {} with error:
 city: Field required
 Please fix the error and try again.

STEP 3
LLM → TOOL
Tool: get_weather_data
Arguments: {'city': 'Bhopal'}

STEP 4
TOOL → LLM
Tool: get_weather_data
Result: {"request": {"type": "City", "query": "Bhopal, India", "language": "en", "unit": "m"}, "location": {"name": "Bhopal", "country": "India", "region": "Madhya Pradesh", "lat": "23.267", "lon": "77.400", "timezone_id": "Asia/Kolkata", "localtime": "2026-08-23 15:56", "localtime_epoch": 1787500560, "utc_offset": "5.50"}, "current": {"observation_time": "10:26 AM", "temperature": 25, "weather_code": 353, "weather_icons": ["https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0009_light_rain_showers.png"], "weather_descriptions": ["Light rain

In [15]:
response = agent.invoke({
    "messages": [
        (
            "user",
            "hi."
        )
    ]
})

In [16]:
print(response["messages"][-1].content)

Hello! How can I assist you today?
